# Chapter 08 — Multiple and Polynomial Regression

Building on Notebook 01, we now handle **multiple features** and capture
**non-linear relationships** through polynomial expansion — all while staying
within the linear-regression framework.

---
## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

rng = np.random.default_rng(42)

---
## 2. Multiple Linear Regression

### 2.1 The Model

When we have $p$ predictor features, the model becomes:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

In matrix notation:

$$\hat{\mathbf{y}} = \mathbf{X} \boldsymbol{\beta}$$

where $\mathbf{X}$ is the **design matrix** (with a column of ones for the intercept)
and $\boldsymbol{\beta}$ is the coefficient vector.

Each coefficient $\beta_j$ tells you: *holding all other features constant, how much
does $\hat{y}$ change when $x_j$ increases by one unit?*

### 2.2 Generating Multi-Feature Synthetic Data

True relationship:

$$y = 2x_1 - 3x_2 + 0.5x_3 + 10 + \varepsilon$$

In [ ]:
n = 200
TRUE_COEFS = np.array([2.0, -3.0, 0.5])
TRUE_INTERCEPT = 10.0

X_multi = rng.uniform(-5, 5, size=(n, 3))
noise = rng.normal(0, 2, size=n)
y_multi = X_multi @ TRUE_COEFS + TRUE_INTERCEPT + noise

print(f'Feature matrix shape: {X_multi.shape}')
print(f'Target vector shape:  {y_multi.shape}')

### 2.3 The Feature Matrix Concept

Each row of `X_multi` is one observation; each column is one feature.
Let's look at the first few rows to see the structure.

In [ ]:
print('First 5 rows of the feature matrix:')
print(X_multi[:5].round(3))
print(f'\nCorresponding y values: {y_multi[:5].round(3)}')

### 2.4 Fitting Multiple Linear Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42
)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)

print('Estimated coefficients vs true values:')
for i, (est, true) in enumerate(zip(model_multi.coef_, TRUE_COEFS)):
    print(f'  beta_{i+1}: estimated = {est:.4f},  true = {true:.4f}')
print(f'  intercept: estimated = {model_multi.intercept_:.4f},  true = {TRUE_INTERCEPT:.4f}')

### 2.5 Interpreting Multiple Coefficients

- $\beta_1 \approx 2$: increasing $x_1$ by 1 (while $x_2, x_3$ stay fixed) raises $\hat{y}$ by ~2.
- $\beta_2 \approx -3$: increasing $x_2$ by 1 **decreases** $\hat{y}$ by ~3.
- $\beta_3 \approx 0.5$: $x_3$ has a small positive effect.

The **sign** tells you the direction; the **magnitude** tells you the strength.

In [ ]:
y_pred_multi = model_multi.predict(X_test)

print(f'Test R²:   {r2_score(y_test, y_pred_multi):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_multi)):.4f}')

### 2.6 Actual vs Predicted Plot

For multi-feature models we cannot easily plot a regression line in 2-D.
Instead, we plot actual values against predicted values — a perfect model
would have all points on the diagonal.

In [ ]:
plt.scatter(y_test, y_pred_multi, alpha=0.6, edgecolors='k', linewidths=0.5)
lims = [min(y_test.min(), y_pred_multi.min()) - 1,
        max(y_test.max(), y_pred_multi.max()) + 1]
plt.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel('Actual y')
plt.ylabel('Predicted y')
plt.title('Multiple Regression — Actual vs Predicted')
plt.legend()
plt.show()

Points cluster tightly around the diagonal — the model captures the multi-feature
relationship well.

### 2.7 Coefficient Bar Chart

A useful way to compare feature importance in a linear model is a bar chart
of coefficients.

In [ ]:
feature_names = ['x1', 'x2', 'x3']
colors = ['green' if c > 0 else 'red' for c in model_multi.coef_]

plt.barh(feature_names, model_multi.coef_, color=colors, edgecolor='k')
plt.xlabel('Coefficient value')
plt.title('Multiple Regression Coefficients')
plt.axvline(0, color='k', linewidth=0.8)
plt.show()

---
## 3. Polynomial Regression

### 3.1 The Idea

What if the relationship between $x$ and $y$ is curved, not straight?

Polynomial regression fits:

$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$$

This is **still linear regression** — it is linear *in the coefficients*.
We simply create new features ($x^2, x^3, \ldots$) and feed them to
`LinearRegression`. sklearn's `PolynomialFeatures` does this transformation for us.

### 3.2 Generating Non-Linear Synthetic Data

True relationship:

$$y = 0.5x^3 - 2x^2 + x + 3 + \varepsilon$$

In [ ]:
n_poly = 150
x_poly = rng.uniform(-3, 4, size=n_poly)
y_poly = 0.5 * x_poly**3 - 2 * x_poly**2 + x_poly + 3 + rng.normal(0, 3, size=n_poly)

X_poly = x_poly.reshape(-1, 1)

plt.scatter(x_poly, y_poly, alpha=0.5, edgecolors='k', linewidths=0.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Non-Linear Synthetic Data')
plt.show()

### 3.3 How PolynomialFeatures Works

For a single feature $x$ and degree 3, it creates: $[1, x, x^2, x^3]$.

In [ ]:
pf = PolynomialFeatures(degree=3, include_bias=False)
X_demo = np.array([[2], [3], [4]])
X_demo_poly = pf.fit_transform(X_demo)

print('Original features:')
print(X_demo)
print(f'\nFeature names: {pf.get_feature_names_out()}')
print(f'\nPolynomial features:')
print(X_demo_poly)

Each row now has $x$, $x^2$, and $x^3$. The regression is still linear in these
expanded features, but the resulting curve in the original $x$-space is a polynomial.

### 3.4 Fitting Polynomial Regression Manually

In [ ]:
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_poly, y_poly, test_size=0.2, random_state=42
)

# Transform features
poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_train_p3 = poly3.fit_transform(X_train_p)
X_test_p3 = poly3.transform(X_test_p)

# Fit
model_p3 = LinearRegression()
model_p3.fit(X_train_p3, y_train_p)

print(f'Coefficients: {model_p3.coef_.round(4)}')
print(f'Intercept:    {model_p3.intercept_:.4f}')
print(f'\nTest R²:  {r2_score(y_test_p, model_p3.predict(X_test_p3)):.4f}')

### 3.5 Using Pipelines (the Clean Way)

A `Pipeline` chains preprocessing and modelling into a single object.
This is cleaner, avoids data leakage, and is easier to use with cross-validation.

In [ ]:
pipe_poly3 = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('reg', LinearRegression())
])

pipe_poly3.fit(X_train_p, y_train_p)

print(f'Pipeline coefficients: {pipe_poly3.named_steps["reg"].coef_.round(4)}')
print(f'Pipeline intercept:    {pipe_poly3.named_steps["reg"].intercept_:.4f}')
print(f'Pipeline test R²:      {pipe_poly3.score(X_test_p, y_test_p):.4f}')

The pipeline produces identical results but wraps the entire workflow into one call.

### 3.6 Visualising the Polynomial Fit

In [ ]:
x_plot = np.linspace(-3, 4, 300).reshape(-1, 1)
y_plot = pipe_poly3.predict(x_plot)

plt.scatter(x_poly, y_poly, alpha=0.4, edgecolors='k', linewidths=0.5, label='Data')
plt.plot(x_plot, y_plot, 'r-', linewidth=2, label='Degree-3 polynomial fit')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Polynomial Regression (degree 3)')
plt.legend()
plt.show()

The cubic curve captures the shape of the data far better than a straight line would.

---
## 4. Interaction Terms

### 4.1 What Are Interaction Terms?

With two features $x_1$ and $x_2$, `PolynomialFeatures(degree=2)` creates:

$$[x_1,\; x_2,\; x_1^2,\; x_1 x_2,\; x_2^2]$$

The **interaction term** $x_1 x_2$ captures the idea that the effect of $x_1$ on $y$
may depend on the value of $x_2$ (and vice versa).

For example: the effect of study hours on exam score might depend on sleep quality.

In [ ]:
# Demonstration with 2 features
X_two = np.array([[1, 2], [3, 4], [5, 6]])
pf2 = PolynomialFeatures(degree=2, include_bias=False)
X_two_poly = pf2.fit_transform(X_two)

print(f'Feature names: {pf2.get_feature_names_out()}')
print(f'\nOriginal:\n{X_two}')
print(f'\nExpanded:\n{X_two_poly}')

### 4.2 Interaction-Only Mode

If you only want the interaction terms (no squared terms), set `interaction_only=True`.

In [ ]:
pf_interact = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_interact = pf_interact.fit_transform(X_two)

print(f'Feature names: {pf_interact.get_feature_names_out()}')
print(f'\nExpanded (interaction only):\n{X_interact}')

### 4.3 Synthetic Data with an Interaction Effect

$$y = 3x_1 + 2x_2 + 4 x_1 x_2 + 5 + \varepsilon$$

Here the interaction coefficient is 4 — the combined effect is significant.

In [ ]:
n_int = 300
x1 = rng.uniform(-2, 2, size=n_int)
x2 = rng.uniform(-2, 2, size=n_int)
y_int = 3 * x1 + 2 * x2 + 4 * x1 * x2 + 5 + rng.normal(0, 1.5, size=n_int)

X_int = np.column_stack([x1, x2])
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_int, y_int, test_size=0.2, random_state=42
)

# Model WITHOUT interaction
model_no_int = LinearRegression().fit(X_train_i, y_train_i)
r2_no_int = r2_score(y_test_i, model_no_int.predict(X_test_i))

# Model WITH interaction
pipe_int = Pipeline([
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('reg', LinearRegression())
])
pipe_int.fit(X_train_i, y_train_i)
r2_with_int = r2_score(y_test_i, pipe_int.predict(X_test_i))

print(f'Without interaction: R² = {r2_no_int:.4f}')
print(f'With interaction:    R² = {r2_with_int:.4f}')
print(f'\nInteraction model coefficients: {pipe_int.named_steps["reg"].coef_.round(4)}')
print(f'Feature names: {pipe_int.named_steps["poly"].get_feature_names_out()}')

Including the interaction term dramatically improves the model because the true
data-generating process contains a strong $x_1 x_2$ effect.

---
## 5. The Danger of High-Degree Polynomials (Overfitting)

A polynomial of sufficiently high degree can pass through every training point.
But this is **not** a good model — it memorises noise rather than learning the
underlying pattern.

### 5.1 Overfitting Demonstration

We fit polynomials of degrees 1, 3, 5, 10, and 15 to the same dataset.

In [ ]:
# Small dataset to make overfitting obvious
n_small = 25
x_small = rng.uniform(-3, 3, size=n_small)
y_small = 0.5 * x_small**3 - 2 * x_small**2 + x_small + 3 + rng.normal(0, 3, size=n_small)

X_small = x_small.reshape(-1, 1)
x_plot_s = np.linspace(-3.5, 3.5, 500).reshape(-1, 1)

degrees = [1, 3, 5, 10, 15]
fig, axes = plt.subplots(1, len(degrees), figsize=(20, 4), sharey=True)

for ax, d in zip(axes, degrees):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('reg', LinearRegression())
    ])
    pipe.fit(X_small, y_small)
    y_hat = pipe.predict(x_plot_s)
    
    train_r2 = pipe.score(X_small, y_small)
    
    ax.scatter(x_small, y_small, alpha=0.6, edgecolors='k', linewidths=0.5)
    ax.plot(x_plot_s, y_hat, 'r-', linewidth=2)
    ax.set_title(f'Degree {d}\nTrain R² = {train_r2:.3f}')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(y_small.min() - 10, y_small.max() + 10)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
plt.suptitle('Polynomial Fits of Increasing Degree', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**Observations:**

- **Degree 1** (underfitting): too simple, misses the curve.
- **Degree 3**: captures the shape nicely — matches the true cubic.
- **Degree 5**: slightly wiggly but still reasonable.
- **Degree 10+** (overfitting): wild oscillations between data points. The model
  has essentially memorised the training data, including the noise.

### 5.2 Train vs Test Error Across Degrees

In [ ]:
# Use the larger polynomial dataset with a proper train/test split
degrees_range = range(1, 16)
train_r2_list = []
test_r2_list = []

for d in degrees_range:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('reg', LinearRegression())
    ])
    pipe.fit(X_train_p, y_train_p)
    train_r2_list.append(pipe.score(X_train_p, y_train_p))
    test_r2_list.append(pipe.score(X_test_p, y_test_p))

plt.plot(list(degrees_range), train_r2_list, 'o-', label='Train R²')
plt.plot(list(degrees_range), test_r2_list, 's-', label='Test R²')
plt.xlabel('Polynomial Degree')
plt.ylabel('R² Score')
plt.title('Train vs Test R² — Overfitting as Degree Increases')
plt.legend()
plt.axvline(3, color='gray', linestyle='--', alpha=0.5, label='True degree')
plt.legend()
plt.show()

The classic overfitting signature:

- **Train R²** keeps increasing (or stays near 1) as complexity grows.
- **Test R²** peaks around the true degree and then drops or becomes unstable.

The gap between train and test performance is the hallmark of overfitting.

---
## 6. Comparing Multiple Polynomial Fits Visually

In [ ]:
x_vis = np.linspace(-3, 4, 400).reshape(-1, 1)
selected_degrees = [1, 2, 3, 5, 8]

plt.figure(figsize=(10, 6))
plt.scatter(x_poly, y_poly, alpha=0.3, edgecolors='k', linewidths=0.3, label='Data', color='gray')

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(selected_degrees)))
for d, color in zip(selected_degrees, colors):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=d, include_bias=False)),
        ('reg', LinearRegression())
    ])
    pipe.fit(X_train_p, y_train_p)
    y_vis = pipe.predict(x_vis)
    r2 = pipe.score(X_test_p, y_test_p)
    plt.plot(x_vis, y_vis, linewidth=2, color=color, label=f'Degree {d} (test R²={r2:.3f})')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Polynomial Fits of Different Degrees')
plt.legend(loc='upper left')
plt.ylim(y_poly.min() - 10, y_poly.max() + 10)
plt.show()

---
## 7. Pipeline Details

### 7.1 Why Use Pipelines?

1. **Convenience**: one `.fit()` / `.predict()` call for the whole workflow.
2. **Safety**: transformations are fitted on training data and applied consistently
   to test data — no data leakage.
3. **Compatibility**: pipelines work seamlessly with `GridSearchCV` and
   `cross_val_score` (covered in Notebook 03).

### 7.2 Inspecting Pipeline Steps

In [ ]:
pipe_example = Pipeline([
    ('poly', PolynomialFeatures(degree=3, include_bias=False)),
    ('reg', LinearRegression())
])
pipe_example.fit(X_train_p, y_train_p)

print('Pipeline steps:')
for name, step in pipe_example.named_steps.items():
    print(f'  {name}: {step}')

print(f'\nAccessing coefficients: {pipe_example.named_steps["reg"].coef_.round(4)}')
print(f'Accessing feature names: {pipe_example.named_steps["poly"].get_feature_names_out()}')

### 7.3 Pipeline Parameters for Grid Search

Pipeline parameters are accessed with the `stepname__parameter` syntax.
This will be crucial for `GridSearchCV` in Notebook 03.

In [ ]:
print('Pipeline parameters:')
for param, value in pipe_example.get_params().items():
    if '__' in param:  # Only show step-specific parameters
        print(f'  {param}: {value}')

---
## 8. Feature Explosion Warning

With multiple features, the number of polynomial features grows rapidly.
For $p$ features and degree $d$, the number of terms is $\binom{p + d}{d}$.

In [ ]:
from math import comb

print(f'{"Features":>10} {"Degree":>8} {"Poly features":>15}')
print('-' * 35)
for p in [2, 5, 10, 20]:
    for d in [2, 3, 5]:
        n_features = comb(p + d, d) - 1  # exclude bias
        print(f'{p:>10} {d:>8} {n_features:>15}')

With 20 features and degree 5, you get over 53,000 polynomial features!
This is a recipe for massive overfitting — regularization (Notebook 03) becomes
essential.

---
## 9. Summary

| Concept | Key Takeaway |
|---------|-------------|
| Multiple regression | Add more features as columns in $\mathbf{X}$ — same OLS approach |
| Coefficient interpretation | Each $\beta_j$ = effect of $x_j$ holding others constant |
| Polynomial features | Create $x^2, x^3, \ldots$ to capture non-linear relationships |
| Interaction terms | $x_1 x_2$ captures joint effects |
| Overfitting | High-degree polynomials memorise noise — test R² degrades |
| Pipelines | Chain preprocessing + model; cleaner code, no data leakage |
| Feature explosion | Polynomial features grow combinatorially with degree and feature count |

**Next:** Notebook 03 introduces **regularization** (Ridge, Lasso, ElasticNet) and
**cross-validation** to combat overfitting systematically.